In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Boolean Network Attractor Simulation from Pathway XML
======================================================
This script builds a directed graph from an XML file (e.g., oncogene_pathways.xml),
extracts a subnetwork around high‑PageRank seed nodes, translates edge types
(activation/inhibition) into Boolean update rules, simulates attractors via
random initial states, and analyzes phenotypic markers (proliferation, apoptosis,
viral) as well as the effects of perturbations (e.g., knocking out viral proteins).

Usage:
    python 09_boolean_network_attractor_simulation.py
"""

import re
import random
import xml.etree.ElementTree as ET
from collections import Counter

import networkx as nx
import pandas as pd
import numpy as np

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# ----------------------------------------------------------------------
# 1. Build directed graph from XML
# ----------------------------------------------------------------------
def build_graph_from_xml(xml_file):
    """Parse XML and return a directed graph with edge types."""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    G = nx.DiGraph()
    for pathway in root.findall('pathway'):
        for node in pathway.findall('.//node'):
            node_name = node.get('name')
            if node_name:
                G.add_node(node_name)
        for edge in pathway.findall('.//edge'):
            start = edge.find('startNode').get('node')
            end = edge.find('endNode').get('node')
            etype = edge.get('type')
            if start and end:
                G.add_edge(start, end, type=etype)
    return G

# ----------------------------------------------------------------------
# 2. Helper: safe name for Boolean rules (alphanumeric + underscore)
# ----------------------------------------------------------------------
def safe_name(name):
    """Convert a node name to a valid Python identifier."""
    s = re.sub(r'[^a-zA-Z0-9]', '_', name)
    if s and s[0].isdigit():
        s = 'n_' + s
    return s

# ----------------------------------------------------------------------
# 3. Subnetwork extraction based on PageRank seeds
# ----------------------------------------------------------------------
def get_top_pagerank_seeds(G, top_k=10):
    """Compute PageRank and return top K nodes as seeds."""
    pr = nx.pagerank(G)
    sorted_nodes = sorted(pr.items(), key=lambda x: x[1], reverse=True)
    seeds = [node for node, _ in sorted_nodes[:top_k]]
    return seeds

def expand_subnetwork(G, seeds, max_steps=2, max_nodes=400):
    """Expand seeds to include predecessors and successors up to max_steps."""
    sub_nodes = set(seeds)
    for node in seeds:
        # Predecessors
        for pred in G.predecessors(node):
            sub_nodes.add(pred)
            if max_steps >= 2:
                for ppred in G.predecessors(pred):
                    sub_nodes.add(ppred)
        # Successors
        for succ in G.successors(node):
            sub_nodes.add(succ)
            if max_steps >= 2:
                for ssucc in G.successors(succ):
                    sub_nodes.add(ssucc)

    if len(sub_nodes) > max_nodes:
        # Fallback to 1‑step neighbors
        sub_nodes = set(seeds)
        for node in seeds:
            for pred in G.predecessors(node):
                sub_nodes.add(pred)
            for succ in G.successors(node):
                sub_nodes.add(succ)
    return G.subgraph(sub_nodes).copy()

# ----------------------------------------------------------------------
# 4. Boolean rule generation from edge types
# ----------------------------------------------------------------------
def generate_boolean_rules(G_sub, node_map):
    """
    Generate Boolean rules (as strings) for each node based on incoming edges.
    Edge type mapping:
        UREG, LIG, INTSB -> activation (OR)
        INTNSB -> activation but requires AND of all such inputs
        DREGC, DREGNC -> inhibition (NOT OR)
    Nodes without incoming edges are set to 0.
    """
    rules = {}
    for node in G_sub.nodes():
        in_edges = list(G_sub.in_edges(node, data=True))
        if not in_edges:
            rules[node_map[node]] = "0"
            continue

        simple_activators = []   # OR
        complex_activators = []  # AND
        inhibitors = []          # NOT (OR)

        for u, v, data in in_edges:
            etype = data.get('type', '')
            if etype in ['UREG', 'LIG']:
                simple_activators.append(node_map[u])
            elif etype in ['DREGC', 'DREGNC']:
                inhibitors.append(node_map[u])
            elif etype == 'INTNSB':
                complex_activators.append(node_map[u])
            elif etype == 'INTSB':
                simple_activators.append(node_map[node])   # self‑activation
            # INTDIS etc. ignored

        expr_parts = []
        if simple_activators:
            expr_parts.append("(" + " or ".join(simple_activators) + ")")
        if complex_activators:
            expr_parts.append("(" + " and ".join(complex_activators) + ")")
        if inhibitors:
            expr_parts.append("not (" + " or ".join(inhibitors) + ")")

        if not expr_parts:
            rules[node_map[node]] = "0"
        elif len(expr_parts) == 1:
            rules[node_map[node]] = expr_parts[0]
        else:
            rules[node_map[node]] = " and ".join(expr_parts)
    return rules

# ----------------------------------------------------------------------
# 5. Boolean network simulator
# ----------------------------------------------------------------------
class BooleanNetwork:
    def __init__(self, rules):
        self.rules = rules
        self.nodes = list(rules.keys())
        self.funcs = {}
        for node, expr in rules.items():
            try:
                code = f"lambda {', '.join(self.nodes)}: {expr}"
                func = eval(code)
                self.funcs[node] = func
            except Exception as e:
                print(f"Compilation error for {node}: {expr} -> {e}")
                self.funcs[node] = lambda **kwargs: 0
        self.node_order = self.nodes

    def update(self, state):
        kwargs = {n: state[n] for n in self.node_order}
        new_state = {}
        for node in self.nodes:
            new_state[node] = int(self.funcs[node](**kwargs))
        return new_state

    def simulate(self, init_state, max_steps=100):
        state = init_state.copy()
        for _ in range(max_steps):
            new_state = self.update(state)
            if new_state == state:
                return state
            state = new_state
        return state   # not stabilized

def find_attractors_with_frequency(bn, n_samples=500, max_steps=50):
    """Sample random initial states and count resulting attractors."""
    attractor_counter = Counter()
    for _ in range(n_samples):
        init = {n: random.randint(0, 1) for n in bn.nodes}
        att = bn.simulate(init, max_steps)
        attractor = tuple(att[n] for n in bn.nodes)
        attractor_counter[attractor] += 1
    return attractor_counter

# ----------------------------------------------------------------------
# 6. Phenotype classification
# ----------------------------------------------------------------------
def classify_attractor(state_tuple, node_order, prolif_set, apop_set, virus_set):
    state_dict = dict(zip(node_order, state_tuple))
    prolif = any(state_dict.get(m, 0) for m in prolif_set)
    apop = any(state_dict.get(m, 0) for m in apop_set)
    virus = any(state_dict.get(m, 0) for m in virus_set)
    return (prolif, apop, virus)

# ----------------------------------------------------------------------
# 7. Perturbation analysis
# ----------------------------------------------------------------------
def perturbation_analysis(bn, fixed_nodes, fixed_values, n_samples=500, max_steps=50):
    fixed_dict = dict(zip(fixed_nodes, fixed_values))
    attractor_counter = Counter()
    for _ in range(n_samples):
        init = {n: random.randint(0, 1) for n in bn.nodes}
        for node, val in fixed_dict.items():
            init[node] = val
        att = bn.simulate(init, max_steps)
        attractor = tuple(att[n] for n in bn.nodes)
        attractor_counter[attractor] += 1
    return attractor_counter

# ----------------------------------------------------------------------
# 8. Critical node detection (only for nodes active in the reference attractor)
# ----------------------------------------------------------------------
def is_critical_node(bn, node, ref_state_dict, ref_tuple, n_samples=20, max_steps=50):
    """Fix node to opposite of its reference value; if any simulation diverges, node is critical."""
    fixed_val = 1 - ref_state_dict[node]
    fixed_dict = {node: fixed_val}
    node_order = bn.node_order

    for _ in range(n_samples):
        init = {n: random.randint(0, 1) for n in node_order}
        init.update(fixed_dict)
        att = bn.simulate(init, max_steps)
        new_tuple = tuple(att[n] for n in node_order)
        if new_tuple != ref_tuple:
            return True
    return False

# ----------------------------------------------------------------------
# 9. Main pipeline
# ----------------------------------------------------------------------
def main():
    xml_file = "oncogene_pathways.xml"
    print("Loading graph from XML...")
    G = build_graph_from_xml(xml_file)
    print(f"Full graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Seed nodes based on PageRank
    seeds = get_top_pagerank_seeds(G, top_k=10)
    print("Seed nodes (PageRank top 10):")
    print(seeds)

    # Expand subnetwork (2‑step neighborhood)
    G_sub = expand_subnetwork(G, seeds, max_steps=2, max_nodes=400)
    print(f"Subnetwork: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")

    # Safe node names
    node_map = {n: safe_name(n) for n in G_sub.nodes()}
    reverse_map = {v: k for k, v in node_map.items()}

    # Generate Boolean rules
    rules = generate_boolean_rules(G_sub, node_map)
    print("\nBoolean rules generated. First 5 examples:")
    for i, (k, v) in enumerate(list(rules.items())[:5]):
        print(f"{k} := {v}")

    # Save rules to file
    with open("boolean_network_rules_safe.txt", "w") as f:
        for node, rule in rules.items():
            f.write(f"{node} := {rule}\n")
    print("\nBoolean rules saved to boolean_network_rules_safe.txt")

    # Build Boolean network and find attractors
    bn = BooleanNetwork(rules)
    print("\nSampling attractors (500 random initial states)...")
    attractor_counter = find_attractors_with_frequency(bn, n_samples=500, max_steps=50)
    print(f"Found {len(attractor_counter)} distinct attractors")

    # Display most common attractors
    most_common = attractor_counter.most_common(5)
    for i, (att, freq) in enumerate(most_common):
        state_dict = dict(zip(bn.node_order, att))
        active = [n for n, v in state_dict.items() if v == 1]
        print(f"Attractor {i+1}: frequency {freq}, active nodes {len(active)}")
        if len(active) <= 30:
            print(f"  Active: {', '.join(active)}")

    # Reference attractor (most frequent)
    if attractor_counter:
        ref_attractor = most_common[0][0]
        ref_state = dict(zip(bn.node_order, ref_attractor))
        print(f"\nReference attractor active nodes: {sum(ref_state.values())}")
    else:
        print("No attractor found.")
        return

    # Phenotype markers (using safe names)
    proliferation_markers = ['ERK', 'PI3K', 'RAS', 'RAF', 'MEK', 'AP1', 'CCND1', 'MYC']
    apoptosis_markers = ['BAX']
    virus_markers = ['TAX', 'P30II', 'HTLV_1_Tax']

    available_prolif = [m for m in proliferation_markers if m in rules]
    available_apop = [m for m in apoptosis_markers if m in rules]
    available_virus = [m for m in virus_markers if m in rules]
    print(f"\nAvailable proliferation markers: {available_prolif}")
    print(f"Available apoptosis markers: {available_apop}")
    print(f"Available virus markers: {available_virus}")

    # Classify attractors
    class_counter = Counter()
    for att, freq in attractor_counter.items():
        cls = classify_attractor(att, bn.node_order, available_prolif, available_apop, available_virus)
        class_counter[cls] += freq
    print("\nAttractor classification (weighted by frequency):")
    for (prolif, apop, virus), total_freq in class_counter.items():
        print(f"Proliferation={prolif}, Apoptosis={apop}, Virus={virus}: {total_freq} times")

    # Perturbation: knock out all viral markers
    if available_virus:
        print(f"\nPerturbation: knockout virus proteins {available_virus} (set to 0)")
        att_pert = perturbation_analysis(bn, available_virus, [0]*len(available_virus), n_samples=500)
        print(f"After knockout, found {len(att_pert)} attractors")
        class_pert = Counter()
        for att, freq in att_pert.items():
            cls = classify_attractor(att, bn.node_order, available_prolif, available_apop, available_virus)
            class_pert[cls] += freq
        print("Attractor types after knockout:")
        for (prolif, apop, virus), total_freq in class_pert.items():
            print(f"  Proliferation={prolif}, Apoptosis={apop}, Virus={virus}: {total_freq} times")
        virus_true = sum(freq for (_, _, virus), freq in class_pert.items() if virus)
        print(f"  Attractors with virus=True: {virus_true} (should be 0)")

    # Additional perturbation: knock out TAX and P30II individually
    if 'TAX' in rules and 'P30II' in rules:
        print("\nAdditional perturbation: knock out TAX and P30II together")
        att_tax = perturbation_analysis(bn, ['TAX', 'P30II'], [0, 0], n_samples=500)
        class_tax = Counter()
        for att, freq in att_tax.items():
            cls = classify_attractor(att, bn.node_order, available_prolif, available_apop, available_virus)
            class_tax[cls] += freq
        print("Attractor types after TAX+P30II knockout:")
        for (prolif, apop, virus), total_freq in class_tax.items():
            print(f"  Proliferation={prolif}, Apoptosis={apop}, Virus={virus}: {total_freq} times")

    # Critical node detection (only for nodes active in reference attractor)
    active_nodes_safe = [n for n, v in ref_state.items() if v == 1]
    print(f"\nCritical node detection on {len(active_nodes_safe)} active nodes...")
    critical_nodes = []
    for i, node in enumerate(active_nodes_safe):
        if is_critical_node(bn, node, ref_state, ref_attractor, n_samples=20, max_steps=50):
            critical_nodes.append(node)
            print(f"  {node}: critical")
        else:
            print(f"  {node}: non‑critical")
        if (i+1) % 20 == 0:
            print(f"  Progress: {i+1}/{len(active_nodes_safe)}")
    print(f"\nCritical nodes found: {len(critical_nodes)}")
    if critical_nodes:
        print("Critical nodes (safe names):", critical_nodes)
        original_names = [reverse_map.get(n, n) for n in critical_nodes]
        print("Critical nodes (original):", original_names)
    else:
        print("No critical nodes detected (network is robust).")

    # Optional: detailed analysis for ERK if present
    if 'ERK' in rules:
        print("\n=== ERK rule analysis ===")
        print(f"ERK rule: {rules['ERK']}")
        if 'ERK' in ref_state:
            print(f"ERK value in reference attractor: {ref_state['ERK']}")
        if 'PTP' in rules:
            print(f"PTP rule: {rules['PTP']}")
            if 'PTP' in ref_state:
                print(f"PTP value in reference attractor: {ref_state['PTP']}")
        # Parse activation part (simple heuristic)
        act_match = re.search(r'\((.*?)\)\s+and\s+not', rules['ERK'])
        if act_match:
            activators_expr = act_match.group(1)
            activator_names = re.findall(r'[a-zA-Z0-9_]+', activators_expr)
            if activator_names:
                print("ERK activation condition involves:", activator_names)
                for name in activator_names:
                    if name in ref_state:
                        print(f"  {name} = {ref_state[name]}")
    print("\nAnalysis completed.")

if __name__ == "__main__":
    main()